# Kinds of features

A model learns from a table. Each row is one example; one column, the **target**, is what the model must learn to predict, and the others are its **features**. Before deciding how to prepare a column, the first question is what kind of value it holds: a quantity, a label, a moment in time, a piece of text, a name for the row. Each kind needs a different preparation, and the type that pandas gives a column doesn't answer the question.

## A small table

Six cars for sale, with one column of each kind. The target is the price:

In [ ]:
import pandas as pd

cars = pd.DataFrame({
    "car_id": [1041, 1042, 1043, 1044, 1045, 1046],
    "registered": pd.to_datetime(["2016-03-14", "2019-07-01", "2012-11-23", "2021-02-09", "2018-05-30", "2015-09-17"]),
    "km": [98_500, 41_200, 187_000, 12_900, 63_300, 120_750],
    "doors": [5, 3, 5, 5, 3, 5],
    "fuel": ["diesel", "petrol", "diesel", "electric", "petrol", "hybrid"],
    "condition": ["good", "excellent", "fair", "excellent", "good", "poor"],
    "automatic": [False, False, False, True, True, False],
    "postcode": [15701, 36201, 15001, 27002, 32003, 15701],
    "description": [
        "One owner, full service history",
        "City car, new tyres",
        "High mileage, runs well",
        "Under warranty, home charger included",
        "Small dent on the rear door",
        "Needs a new clutch",
    ],
    "price": [9_800, 11_500, 3_900, 24_900, 14_200, 4_500],
})
cars

,car_id,registered,km,doors,fuel,condition,automatic,postcode,description,price
0,1041,2016-03-14,98500,5,diesel,good,False,15701,"One owner, full service history",9800
1,1042,2019-07-01,41200,3,petrol,excellent,False,36201,"City car, new tyres",11500
2,1043,2012-11-23,187000,5,diesel,fair,False,15001,"High mileage, runs well",3900
3,1044,2021-02-09,12900,5,electric,excellent,True,27002,"Under warranty, home charger included",24900
4,1045,2018-05-30,63300,3,petrol,good,True,32003,Small dent on the rear door,14200
5,1046,2015-09-17,120750,5,hybrid,poor,False,15701,Needs a new clutch,4500


Each column is one of these kinds:

| Kind | In the table | What it is |
| --- | --- | --- |
| **Continuous number** | `km`, `price` | A quantity that can take any value in a range. Differences mean something: 10,000 km more is 10,000 km more, whatever the starting point |
| **Discrete number** | `doors` | A count: whole numbers, but still a quantity, with an order and meaningful differences |
| **Nominal category** | `fuel` | A label from a fixed set, with no order: diesel is neither more nor less than petrol |
| **Ordinal category** | `condition` | A label with an order (poor < fair < good < excellent) but no distances: nothing says that the step from poor to fair equals the step from good to excellent |
| **Boolean** | `automatic` | A category with two values, already a 0 or a 1 |
| **Date or time** | `registered` | A moment. A model can't use it as it is, but it contains features: the car's age, the month, the day of the week |
| **Free text** | `description` | Words, different in every row |
| **Identifier** | `car_id` | A name for the row. It says which car, not anything about it |
| **Code written in digits** | `postcode` | A category whose labels happen to be numbers |

## The type is not the kind

pandas chooses a type for each column from what the values look like, not from what they mean:

In [ ]:
cars.dtypes

car_id                  int64
registered     datetime64[us]
km                      int64
doors                   int64
fuel                      str
condition                 str
automatic                bool
postcode                int64
description               str
price                   int64
dtype: object

`car_id` and `postcode` are stored as integers, like `km` and `doors`. Anything that treats integers as quantities will treat these two as quantities too. `describe` summarises them alongside the real numbers:

In [ ]:
cars.describe().loc[["mean", "min", "max"], ["km", "doors", "postcode", "car_id"]]

,km,doors,postcode,car_id
mean,87275.0,4.333333,23601.5,1043.5
min,12900.0,3.000000,15001.0,1041.0
max,187000.0,5.000000,36201.0,1046.0


The mean of the postcodes is a number that means nothing: postcode 15701 isn't "between" 15001 and 27002 in any useful sense, and a model given this column as a number would learn exactly that kind of meaningless relationship, without any error or warning. The same goes for `car_id`: as a number it would let a model look for a relationship between a car's identifier and its price, and any it found would be chance, or a trace of how the identifiers were assigned (in order of arrival, say), with nothing to say about the next car.

The text columns have the opposite problem: `fuel`, `condition` and `description` all have the type `str`, although the first is a category without an order, the second a category with one, and the third free text.

The kind can be declared. A categorical type with an explicit order makes `condition` sort by its meaning instead of alphabetically:

In [ ]:
condition_order = pd.CategoricalDtype(["poor", "fair", "good", "excellent"], ordered=True)

pd.DataFrame({
    "as text": cars["condition"].sort_values().to_list(),
    "as an ordered category": cars["condition"].astype(condition_order).sort_values().to_list(),
})

,as text,as an ordered category
0,excellent,poor
1,excellent,fair
2,fair,good
3,good,good
4,good,excellent
5,poor,excellent


Sorted as text, "excellent" comes first and "poor" last, by their first letters. Declared as an ordered category, the column sorts from worst to best, and comparisons such as `>= "good"` follow the order. Declaring kinds early, while reading the data, keeps them in one place for pandas, but scikit-learn's encoders don't read the order declared there: `OrdinalEncoder` still sorts the categories alphabetically unless it is given the order again. [Encoding categories](../categorical_encoding.ipynb) turns both kinds of category into numbers.

Since pandas 3, text columns have the type `str`; before, they had the generic type `object`. Code written for older versions often selects the text columns with `select_dtypes(include="object")`. pandas 3 still accepts it, with a warning that it will stop doing so: select them with `include="str"`.

## A real table: the Titanic passengers

The passenger list of the Titanic, which sank in 1912, is one of the most used tables for teaching, for a good reason: almost every kind of column is in it. OpenML, an open repository of datasets for machine learning, publishes it; scikit-learn downloads it the first time and keeps a copy in `data/`:

In [ ]:
from sklearn.datasets import fetch_openml

titanic = fetch_openml("titanic", version=1, as_frame=True, data_home="data")
passengers = titanic.frame
passengers.head()

,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0,0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1,0,"Allison, Miss. Helen Loraine",female,2.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1,0,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1,2,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"


The target is `survived`. A survey of the other columns, with their type, the number of distinct values and the number of missing ones, is the quickest way to start classifying them:

In [ ]:
pd.DataFrame({
    "type": passengers.dtypes.astype(str),
    "distinct values": passengers.nunique(),
    "missing": passengers.isna().sum(),
})

,type,distinct values,missing
pclass,int64,3,0
survived,category,2,0
name,str,1307,0
sex,category,2,0
age,float64,98,263
sibsp,int64,7,0
parch,int64,8,0
ticket,str,929,0
fare,float64,281,1
cabin,str,186,1014


Reading each column with the survey and the variable descriptions of the original dataset (*titanic3*, compiled by Thomas Cason from the Encyclopedia Titanica; `titanic.DESCR` gives its source):

| Column | Type | Kind |
| --- | --- | --- |
| `pclass` | integer | Ordinal category: first, second or third class, "a proxy for socio-economic class", as the description puts it. Treating it as a number assumes that the step from first to second class equals the step from second to third |
| `name` | text | Free text, almost one value per passenger (1,307 for 1,309 rows), so useless as it is; but it contains a title (Mr, Mrs, Miss, Master) that is a category, a feature to extract ([Creating features](creating_features.ipynb)) |
| `sex`, `embarked` | category | Nominal categories (`embarked` is the port: Cherbourg, Queenstown or Southampton) |
| `age`, `fare` | float | Continuous numbers. `age` is missing for 263 passengers ([Missing values](../missing_values.ipynb)) |
| `sibsp`, `parch` | integer | Discrete numbers: siblings or spouses, and parents or children, travelling with the passenger |
| `ticket` | text | A code: 929 distinct values, shared only by people travelling together |
| `cabin` | text | A code whose first letter is the deck; missing for 1,014 passengers |
| `boat`, `body` | text, float | The lifeboat, and the number given to a recovered body. **Both are known only after the sinking** |
| `home.dest` | text | Free text: where the passenger lived and was going |

## Columns that must not be features

Some columns must be left out whatever their kind. The clearest are those that contain the answer. `boat` against `survived`:

In [ ]:
pd.crosstab(passengers["boat"].notna(), passengers["survived"])

survived,0,1
boat,,
False,800,23
True,9,477


Of the 486 passengers with a lifeboat recorded, 477 survived; of the 823 without one, 23 did. A model given `boat` would look almost perfect and be useless: when the prediction would be needed, before anyone boards a lifeboat, the column doesn't exist yet. `body` is worse: it exists only for passengers who died. A feature that is known only after the outcome is a leak of the target ([Data leakage](../../supervised-learning/ml-fundamentals/data_leakage.ipynb)).

Identifiers are the other case. A column with a different value in almost every row, such as `name` or `car_id`, lets a model memorise rows instead of learning something that applies to new ones. `ticket` and `cabin` are codes in between: as they are, they identify groups of passengers, but parts of them (the deck in `cabin`) may be real features.

## Beyond tables

Not every dataset is a table of columns of different kinds:

- **Images** are arrays of numbers of a single kind: each pixel is one brightness (grey) or three (red, green and blue), usually a whole number from 0 to 255. [Matrices and tensors in AI](https://github.com/avidaldo/compare-python-c-efficiency/blob/main/02_matrices_in_ai.ipynb) shows how they are stored; [Scaling images](scaling_images.ipynb) explains why they are scaled differently from a table.
- **Text** must become numbers before a model can use it: by counting the words of each document, or as embeddings, vectors that place texts with similar meanings close together. Language models *(coming)* starts from text as data.

Inside a table, a column of free text such as `description` is the same problem in small.

## What each kind needs

| Kind | Usual preparation | Where |
| --- | --- | --- |
| Continuous number | Nothing for trees; scaling for models based on distances, gradients or penalties; a transformation if its distribution has a long tail | [Scaling numerical features](../feature_scaling.ipynb), [Skewed distributions](skewed_distributions.ipynb) |
| Discrete number | As a continuous number | |
| Nominal category | One-hot encoding, or target encoding when there are many categories | [Encoding categories](../categorical_encoding.ipynb) |
| Ordinal category | Ordinal encoding, with the order given explicitly | [Encoding categories](../categorical_encoding.ipynb) |
| Boolean | Nothing: it already is a 0 or a 1 | |
| Date or time | Extracting numbers from it: age, month, day of the week, time elapsed | [Creating features](creating_features.ipynb) |
| Free text | Extracting features (a title, a length) or representing the whole text as numbers | [Creating features](creating_features.ipynb), Language models *(coming)* |
| Identifier | Dropping it from the features (keeping it to link rows to their source) | |
| Any column known only after the outcome | Dropping it | [Data leakage](../../supervised-learning/ml-fundamentals/data_leakage.ipynb) |

Missing values and extreme values can appear in a column of any kind: [Missing values](../missing_values.ipynb) and [Outliers and capped values](outliers.ipynb).